# Notebook 2 — Keyword & Filtered Search

*Hands-on time: ~30 minutes*

Now that data from **all bids-examples datasets** is indexed, we explore ElasticSearch's
**structured query** capabilities:
1. Match queries (full-text / BM25)
2. Term queries (exact keyword match)
3. Range filters (numeric fields)
4. Bool compound queries (`must` / `should` / `filter`)
5. Aggregations (faceted analytics)

With dozens of datasets spanning different scanners, field strengths, and
institutions, we can see meaningful variation in search results.

**Prerequisites:** Complete Notebook 1.

In [18]:
import os
from elasticsearch import Elasticsearch
import pandas as pd

ES_HOST = os.environ.get("ES_HOST", "http://localhost:9200")
client = Elasticsearch(ES_HOST, request_timeout=120)
INDEX_NAME = "neuroimaging"

assert client.ping(), f"Cannot reach ES at {ES_HOST}"
count = client.count(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' has {count['count']} documents\n")

# Dataset overview — shows the diversity of our corpus
overview = client.search(
    index=INDEX_NAME, size=0,
    aggs={"datasets": {"terms": {"field": "dataset", "size": 50}}}
)
for b in overview["aggregations"]["datasets"]["buckets"]:
    print(f"  {b['key']:30s} {b['doc_count']:4d} docs")


# ── Context-aware column selection ──────────────────────────────────────────
# Each query type benefits from seeing different metadata fields.
# These presets match the fields that matter most for each neuroimaging context.

COLS_STRUCTURAL = ["dataset", "suffix", "subject", "MagneticFieldStrength",
                   "Manufacturer", "ManufacturersModelName", "InversionTime",
                   "FlipAngle", "MRAcquisitionType"]
COLS_FUNCTIONAL = ["dataset", "suffix", "subject", "task", "TaskName",
                   "RepetitionTime", "EchoTime", "MagneticFieldStrength",
                   "Manufacturer"]
COLS_DIFFUSION = ["dataset", "suffix", "subject", "MagneticFieldStrength",
                  "Manufacturer", "PhaseEncodingDirection", "SliceThickness"]
COLS_FIELDMAP = ["dataset", "suffix", "subject", "MagneticFieldStrength",
                 "Manufacturer", "PhaseEncodingDirection", "EchoTime"]
COLS_SCANNER = ["dataset", "suffix", "subject", "Manufacturer",
                "ManufacturersModelName", "MagneticFieldStrength",
                "InstitutionName"]
COLS_DEFAULT = ["dataset", "suffix", "subject", "MagneticFieldStrength",
                "Manufacturer", "TaskName", "description_text"]


def show_hits(response_or_hits, fields=None, k=10, trunc=90):
    """
    Pretty-print search results as a DataFrame.

    Parameters
    ----------
    response_or_hits : ES response dict OR a plain list of hit dicts
    fields  : list of field names to display.  None → COLS_DEFAULT
    k       : maximum rows to show (default 10)
    trunc   : truncate description_text to this many chars (default 90)
    """
    if isinstance(response_or_hits, list):
        hits = response_or_hits
    else:
        hits = response_or_hits["hits"]["hits"]

    rows = []
    for hit in hits[:k]:
        row = {"_score": round(float(hit.get("_score") or 0), 4)}
        src = hit.get("_source", {})
        cols = fields or COLS_DEFAULT
        for f in cols:
            if f == "description_text":
                val = src.get(f, "")
                row[f] = (val[:trunc] + "…") if len(val) > trunc else val
            else:
                row[f] = src.get(f)
        rows.append(row)
    return pd.DataFrame(rows)

Index 'neuroimaging' has 4423 documents

  7t_trt                          439 docs
  ds000117                        395 docs
  ds108                           238 docs
  ds210                           225 docs
  ds006                           220 docs
  ds110                           216 docs
  ds009                           192 docs
  ds007                           158 docs
  ds113b                          156 docs
  ds107                           147 docs
  ds051                           143 docs
  ds114                           140 docs
  ds002                           136 docs
  ds116                           136 docs
  ds000001-fmriprep               128 docs
  ds008                           113 docs
  ds011                           112 docs
  ds109                           105 docs
  ds052                            90 docs
  mrs_biggaba                      84 docs
  ds001                            80 docs
  ds005                            80 docs
  ds102      

---
## 1. Match Queries — Full-Text Search (BM25)

A `match` query analyzes the input text and uses BM25 scoring against
analyzed `text` fields like `description_text` and `SeriesDescription`.

In [20]:
# ── Match query: find all structural brain scans (T1-weighted and T2-weighted)
# BM25 scores higher for documents where "anatomical", "structural", "T1" appear
# in the description_text field.  With many datasets the results show real
# variation: field strength, manufacturer, acquisition type.

response = client.search(
    index=INDEX_NAME,
    query={"match": {"description_text": "anatomical structural T1 brain"}},
    size=10
)

print(f"Total matches: {response['hits']['total']['value']}")
print("Note the spread of datasets, manufacturers, and field strengths →")
show_hits(response, COLS_STRUCTURAL)

Total matches: 1024
Note the spread of datasets, manufacturers, and field strengths →


,_score,dataset,suffix,subject,MagneticFieldStrength,Manufacturer,ManufacturersModelName,InversionTime,FlipAngle,MRAcquisitionType
0,7.9791,mrs_biggaba,T1w,01,3.0,Philips,Achieva,0.865,8.0,3D
1,7.9791,mrs_biggaba,T1w,02,3.0,Philips,Achieva,0.865,8.0,3D
2,7.9791,mrs_biggaba,T1w,03,3.0,Philips,Achieva,0.865,8.0,3D
3,7.9791,mrs_biggaba,T1w,04,3.0,Philips,Achieva,0.865,8.0,3D
4,7.9791,mrs_biggaba,T1w,05,3.0,Philips,Achieva,0.865,8.0,3D
5,7.9791,mrs_biggaba,T1w,06,3.0,Philips,Achieva,0.865,8.0,3D
6,7.9791,mrs_biggaba,T1w,07,3.0,Philips,Achieva,0.865,8.0,3D
7,7.9791,mrs_biggaba,T1w,08,3.0,Philips,Achieva,0.865,8.0,3D
8,7.9791,mrs_biggaba,T1w,09,3.0,Philips,Achieva,0.865,8.0,3D
9,7.9791,mrs_biggaba,T1w,10,3.0,Philips,Achieva,0.865,8.0,3D


In [21]:
# ── Multi-match: search several text fields simultaneously
# 'cross_fields' treats all fields as one concatenated document.
# This catches 'resting state' in TaskName, 'Siemens' in description_text, etc.

response = client.search(
    index=INDEX_NAME,
    query={
        "multi_match": {
            "query": "resting state fMRI Siemens",
            "fields": ["description_text^2", "TaskName^1.5", "SeriesDescription"],
            "type": "cross_fields",
            "operator": "or"
        }
    },
    size=10
)

print(f"Total matches: {response['hits']['total']['value']}")
print("Columns show task + scanner identity — what matters for resting-state →")
show_hits(response, COLS_FUNCTIONAL)

Total matches: 725
Columns show task + scanner identity — what matters for resting-state →


,_score,dataset,suffix,subject,task,TaskName,RepetitionTime,EchoTime,MagneticFieldStrength,Manufacturer
0,9.8728,eeg_rest_fmri,dwi,32,,None,8.300,0.0980,1.5,Siemens
1,9.8728,eeg_rest_fmri,dwi,32,,None,8.300,0.0980,1.5,Siemens
2,9.8728,eeg_rest_fmri,dwi,35,,None,8.300,0.0980,1.5,Siemens
3,9.8728,eeg_rest_fmri,dwi,35,,None,8.300,0.0980,1.5,Siemens
4,9.8728,eeg_rest_fmri,dwi,36,,None,8.300,0.0980,1.5,Siemens
5,9.8728,eeg_rest_fmri,dwi,36,,None,8.300,0.0980,1.5,Siemens
6,9.3983,eeg_rest_fmri,T1w,32,,None,0.011,0.4940,1.5,Siemens
7,9.3983,eeg_rest_fmri,T1w,32,,None,0.011,0.0494,1.5,Siemens
8,9.3983,eeg_rest_fmri,T1w,35,,None,0.011,0.4940,1.5,Siemens
9,9.3983,eeg_rest_fmri,T1w,35,,None,0.011,0.0494,1.5,Siemens


---
## 1b. Highlight — *Why* Did BM25 Return This Result?

The `highlight` API asks ES to return the exact token matches that triggered
each BM25 hit. This is pedagogically critical: it shows students that BM25
is a term-frequency model, not a semantic model, so results depend entirely
on which tokens overlap between query and document.

Highlights are returned as `<em>` tags around matching terms.


In [23]:
# ── Highlight API: see exactly WHY each document scored
# The <em> tags mark the query terms that BM25 matched.
# Notice: BM25 matches 'resting' and 'state' as separate tokens.
# A phrase like 'resting state' as a unit requires a match_phrase query.

response = client.search(
    index=INDEX_NAME,
    query={"match": {"description_text": "resting state BOLD Siemens 3T"}},
    highlight={
        "fields": {
            "description_text": {"number_of_fragments": 2, "fragment_size": 100},
            "TaskName":          {"number_of_fragments": 1}
        },
        "pre_tags":  ["**["],
        "post_tags": ["]**"]
    },
    size=5
)

print(f"Query: 'resting state BOLD Siemens 3T'")
print(f"Hits : {response['hits']['total']['value']}\n")

for hit in response["hits"]["hits"]:
    src = hit["_source"]
    hl = hit.get("highlight", {})
    print(f"  [{src.get('dataset', '?')}] suffix={src.get('suffix', '?')} "
          f"T={src.get('MagneticFieldStrength', '?')} "
          f"mfg={src.get('Manufacturer', '?')} "
          f"score={hit['_score']:.3f}")
    for field, frags in hl.items():
        for frag in frags:
            print(f"    {field}: {frag}")
    print()

# Key insight: BM25 matched 'resting', 'state', 'BOLD', 'Siemens', '3T' as
# independent tokens.  A document scoring high has ALL these tokens present,
# but BM25 does NOT understand their relationship (resting ≠ resting_state_fMRI)

Query: 'resting state BOLD Siemens 3T'
Hits : 2787

  [eeg_rest_fmri] suffix=bold T=1.5 mfg=Siemens score=2.854
    description_text: A **[BOLD]** functional MRI Blood Oxygen Level Dependent scan acquired during the rest task on a 1.5 Tesla
    description_text: **[Siemens]** Avanto  scanner.

  [eeg_rest_fmri] suffix=bold T=1.5 mfg=Siemens score=2.854
    description_text: A **[BOLD]** functional MRI Blood Oxygen Level Dependent scan acquired during the rest task on a 1.5 Tesla
    description_text: **[Siemens]** Avanto  scanner.

  [eeg_rest_fmri] suffix=bold T=1.5 mfg=Siemens score=2.854
    description_text: A **[BOLD]** functional MRI Blood Oxygen Level Dependent scan acquired during the rest task on a 1.5 Tesla
    description_text: **[Siemens]** Avanto  scanner.

  [genetics_ukbb] suffix=bold T=3.0 mfg=Siemens score=2.821
    description_text: A **[BOLD]** functional MRI Blood Oxygen Level Dependent scan acquired during the FaceShapeEmotion task on
    description_text: a 3 Tesl

**Here we see an interesting effect of BM25 lack of knowledge**: the query asks for *3T*, which is understandably the B0 field strength wanted, aka *3 Tesla*. BM25 is unable to **match** that to the *3 Tesla* snippet in the `description_text`, lacking specific **neuroimaging domain knowledge**. It's a good baseline, but it indicates the need for a better model for semantic encoding of neuroimaging metadata.

---
## 2. Term Queries — Exact Match on Keywords

For `keyword` fields, use `term` (single value) or `terms` (multiple values).
These are **exact** matches — no text analysis.

In [24]:
# ── Term query: exact keyword filtering on 'suffix'
# This is NOT a BM25 query — it's a zero-score filter.
# Real score = 1.0 for all results (term queries are binary matches).
# Use this when you know exactly what you want and don't need ranking.

response = client.search(
    index=INDEX_NAME,
    query={"term": {"suffix": "bold"}},
    size=10
)

print(
    f"Total BOLD scans across all datasets: {response['hits']['total']['value']}")
print("Columns show task context + TR/TE — the key parameters for fMRI analysis →")
show_hits(response, COLS_FUNCTIONAL)

Total BOLD scans across all datasets: 2612
Columns show task context + TR/TE — the key parameters for fMRI analysis →


,_score,dataset,suffix,subject,task,TaskName,RepetitionTime,EchoTime,MagneticFieldStrength,Manufacturer
0,0.5267,7t_trt,bold,01,rest,Rest,3.0,0.017,None,None
1,0.5267,7t_trt,bold,01,rest,Rest,3.0,0.017,None,None
2,0.5267,7t_trt,bold,01,rest,Rest,4.0,0.026,None,None
3,0.5267,7t_trt,bold,01,rest,Rest,3.0,0.017,None,None
4,0.5267,7t_trt,bold,01,rest,Rest,3.0,0.017,None,None
5,0.5267,7t_trt,bold,01,rest,Rest,4.0,0.026,None,None
6,0.5267,7t_trt,bold,02,rest,Rest,3.0,0.017,None,None
7,0.5267,7t_trt,bold,02,rest,Rest,3.0,0.017,None,None
8,0.5267,7t_trt,bold,02,rest,Rest,4.0,0.026,None,None
9,0.5267,7t_trt,bold,02,rest,Rest,3.0,0.017,None,None


In [29]:
# ── Term query on dataset field → drill into one study
# ds000117 is a multi-session MEG+fMRI study (118 participants, 3T Siemens).
# With the 'dataset' keyword we can inspect exactly what that one study contains.

response = client.search(
    index=INDEX_NAME,
    query={"term": {"dataset": "ds000117"}},
    size=10
)

print(f"ds000117 scans: {response['hits']['total']['value']} total")
show_hits(response, COLS_SCANNER)

ds000117 scans: 395 total


,_score,dataset,suffix,subject,Manufacturer,ManufacturersModelName,MagneticFieldStrength,InstitutionName
0,2.4146,ds000117,T1w,01,Siemens,TrioTim,3.0,MRC Cognition and Brain Sciences Unit
1,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
2,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
3,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
4,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
5,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
6,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
7,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
8,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN
9,2.4146,ds000117,FLASH,01,NaN,NaN,NaN,NaN


In [31]:
# ── Terms query: match any of a list of keyword values (OR logic)
# Useful for retrieving all 'structural' scans regardless of exact contrast.

response = client.search(
    index=INDEX_NAME,
    query={"terms": {"suffix": ["T1w", "T2w", "dwi", "FLAIR"]}},
    size=10
)

print(f"Structural / diffusion scans: {response['hits']['total']['value']}")
show_hits(response, COLS_STRUCTURAL)

Structural / diffusion scans: 670


,_score,dataset,suffix,subject,MagneticFieldStrength,Manufacturer,ManufacturersModelName,InversionTime,FlipAngle,MRAcquisitionType
0,1.0,2d_mb_pcasl,T1w,1,3.0,Siemens,Prisma_fit,1.0,8.0,3D
1,1.0,7t_trt,T1w,01,NaN,NaN,NaN,NaN,NaN,NaN
2,1.0,7t_trt,T1w,02,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,7t_trt,T1w,03,NaN,NaN,NaN,NaN,NaN,NaN
4,1.0,7t_trt,T1w,04,NaN,NaN,NaN,NaN,NaN,NaN
5,1.0,7t_trt,T1w,05,NaN,NaN,NaN,NaN,NaN,NaN
6,1.0,7t_trt,T1w,06,NaN,NaN,NaN,NaN,NaN,NaN
7,1.0,7t_trt,T1w,07,NaN,NaN,NaN,NaN,NaN,NaN
8,1.0,7t_trt,T1w,08,NaN,NaN,NaN,NaN,NaN,NaN
9,1.0,7t_trt,T1w,09,NaN,NaN,NaN,NaN,NaN,NaN


---
## 3. Range Filters — Numeric Fields

Range queries let you filter by numeric values: TR, TE, field strength, etc.

In [32]:
# ── Range query on RepetitionTime
# fMRI scans have TR 0.5–3.0 s; structural scans can exceed 10 s.
# A range filter lets students see which modalities lie in each TR band.

response = client.search(
    index=INDEX_NAME,
    query={"range": {"RepetitionTime": {"gte": 2.0, "lte": 3.5}}},
    size=10
)

print(
    f"Scans with TR 2-3.5s: {response['hits']['total']['value']} (typical fMRI range)")
show_hits(response, ["dataset", "suffix", "subject", "task", "TaskName",
                     "RepetitionTime", "MagneticFieldStrength", "Manufacturer"])

Scans with TR 2-3.5s: 2626 (typical fMRI range)


,_score,dataset,suffix,subject,task,TaskName,RepetitionTime,MagneticFieldStrength,Manufacturer
0,1.0,2d_mb_pcasl,T1w,1,,NaN,2.5,3.0,Siemens
1,1.0,7t_trt,bold,01,rest,Rest,3.0,NaN,NaN
2,1.0,7t_trt,bold,01,rest,Rest,3.0,NaN,NaN
3,1.0,7t_trt,bold,01,rest,Rest,3.0,NaN,NaN
4,1.0,7t_trt,bold,01,rest,Rest,3.0,NaN,NaN
5,1.0,7t_trt,bold,02,rest,Rest,3.0,NaN,NaN
6,1.0,7t_trt,bold,02,rest,Rest,3.0,NaN,NaN
7,1.0,7t_trt,bold,02,rest,Rest,3.0,NaN,NaN
8,1.0,7t_trt,bold,02,rest,Rest,3.0,NaN,NaN
9,1.0,7t_trt,bold,03,rest,Rest,3.0,NaN,NaN


In [33]:
# ── Distribution of field strengths in the corpus ──────────────────────────
for fs in [1.5, 3.0, 7.0]:
    resp = client.search(index=INDEX_NAME,
                         query={"term": {"MagneticFieldStrength": fs}}, size=0)
    print(f"  {fs:4.1f}T: {resp['hits']['total']['value']:5d} scans")

# Range: only 3T and 7T (research grade)
response = client.search(
    index=INDEX_NAME,
    query={"range": {"MagneticFieldStrength": {"gte": 3.0}}},
    size=10
)
print(f"\n3T+ scans: {response['hits']['total']['value']} total")
print("These are typically research-grade acquisitions; 7T data is rare and valuable →")
show_hits(response, COLS_SCANNER)

   1.5T:    15 scans
   3.0T:   505 scans
   7.0T:     0 scans

3T+ scans: 507 total
These are typically research-grade acquisitions; 7T data is rare and valuable →


,_score,dataset,suffix,subject,Manufacturer,ManufacturersModelName,MagneticFieldStrength,InstitutionName
0,1.0,2d_mb_pcasl,T1w,1,Siemens,Prisma_fit,3.0,None
1,1.0,2d_mb_pcasl,epi,1,Siemens,Prisma_fit,3.0,None
2,1.0,2d_mb_pcasl,epi,1,Siemens,Prisma_fit,3.0,None
3,1.0,2d_mb_pcasl,asl,1,Siemens,Prisma_fit,3.0,None
4,1.0,asl001,T1w,Sub103,GE,DISCOVERY_MR750,3.0,None
5,1.0,asl001,asl,Sub103,GE,DISCOVERY_MR750,3.0,None
6,1.0,asl002,T1w,Sub103,Philips,Achieva,3.0,None
7,1.0,asl002,asl,Sub103,Philips,Achieva,3.0,None
8,1.0,asl002,m0scan,Sub103,Philips,Achieva,3.0,None
9,1.0,asl003,T1w,Sub1,Siemens,TrioTim,3.0,None


---
## 4. Bool Compound Queries

The `bool` query combines clauses:
- **must**: All clauses must match (AND) — affects score
- **should**: At least one should match (OR) — boosts score
- **filter**: Must match — does NOT affect score (faster, cached)
- **must_not**: Must not match (exclusion)

In [35]:
# ── Bool compound query: Siemens 3T resting-state BOLD with TR in a useful range
# This is the most common query pattern in neuroimaging search:
#   "Give me all resting-state fMRI scans from 3T Siemens scanners"

response = client.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"term": {"suffix": "bold"}},
                {"term": {"Manufacturer": "Siemens"}},
                {"term": {"MagneticFieldStrength": 3.0}}
            ],
            "filter": [
                {"range": {"RepetitionTime": {"gte": 1.0, "lte": 3.0}}}
            ],
            "should": [
                {"match": {"TaskName": "rest"}},
                {"match": {"description_text": "resting state"}}
            ],
            "minimum_should_match": 0
        }
    },
    size=10
)

print(
    f"3T Siemens BOLD scans with TR 1-3s: {response['hits']['total']['value']}")
print("should clauses boost resting-state scans to the top →")
show_hits(response, COLS_FUNCTIONAL)

3T Siemens BOLD scans with TR 1-3s: 147
should clauses boost resting-state scans to the top →


,_score,dataset,suffix,subject,task,TaskName,RepetitionTime,EchoTime,MagneticFieldStrength,Manufacturer
0,4.6413,atlas-4S,bold,01,rest,rest eyes open,1.761,0.03,3.0,Siemens
1,4.6413,atlas-4S,bold,01,rest,rest eyes open,1.761,0.03,3.0,Siemens
2,4.6413,atlas-4S,bold,01,rest,rest eyes open,1.761,0.03,3.0,Siemens
3,1.9105,ds000117,bold,01,facerecognition,facerecognition,2.000,0.03,3.0,Siemens
4,1.9105,ds000117,bold,01,facerecognition,facerecognition,2.000,0.03,3.0,Siemens
5,1.9105,ds000117,bold,01,facerecognition,facerecognition,2.000,0.03,3.0,Siemens
6,1.9105,ds000117,bold,01,facerecognition,facerecognition,2.000,0.03,3.0,Siemens
7,1.9105,ds000117,bold,01,facerecognition,facerecognition,2.000,0.03,3.0,Siemens
8,1.9105,ds000117,bold,01,facerecognition,facerecognition,2.000,0.03,3.0,Siemens
9,1.9105,ds000117,bold,01,facerecognition,facerecognition,2.000,0.03,3.0,Siemens


In [36]:
# Cross-dataset: everything EXCEPT ds001 (show only metadata-rich datasets)
response = client.search(
    index=INDEX_NAME,
    query={
        "bool": {
            "must_not": [
                {"term": {"dataset": "ds001"}}
            ]
        }
    },
    size=5
)

print(f"Non-ds001 scans: {response['hits']['total']['value']}")
show_hits(response, ["dataset", "subject", "suffix",
          "Manufacturer", "MagneticFieldStrength", "InstitutionName"])

Non-ds001 scans: 4343


,_score,dataset,subject,suffix,Manufacturer,MagneticFieldStrength,InstitutionName
0,0.0,2d_mb_pcasl,1,T1w,Siemens,3.0,None
1,0.0,2d_mb_pcasl,1,epi,Siemens,3.0,None
2,0.0,2d_mb_pcasl,1,epi,Siemens,3.0,None
3,0.0,2d_mb_pcasl,1,asl,Siemens,3.0,None
4,0.0,7t_trt,01,T1map,NaN,NaN,None


---
## 5. Aggregations — Analytics & Faceted Counts

Aggregations let you compute statistics over your indexed data,
similar to SQL `GROUP BY`.

In [37]:
# Count documents by dataset (top-level view)
response = client.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_dataset": {
            "terms": {"field": "dataset"},
            "aggs": {
                "suffixes": {"terms": {"field": "suffix"}}
            }
        }
    }
)

print("Documents by dataset and scan type:")
for ds in response["aggregations"]["by_dataset"]["buckets"]:
    suffixes = ", ".join(
        f"{s['key']}({s['doc_count']})" for s in ds["suffixes"]["buckets"])
    print(f"  {ds['key']:25s} {ds['doc_count']:4d} docs — {suffixes}")

Documents by dataset and scan type:
  7t_trt                     439 docs — bold(132), magnitude1(88), phasediff(88), magnitude2(87), T1map(22), T1w(22)
  ds000117                   395 docs — FLASH(224), bold(144), T1w(16), dwi(11)
  ds108                      238 docs — bold(204), T1w(34)
  ds210                      225 docs — bold(225)
  ds006                      220 docs — bold(164), T1w(28), inplaneT2(28)
  ds110                      216 docs — bold(180), T1w(18), inplaneT2(18)
  ds009                      192 docs — bold(144), T1w(24), inplaneT2(24)
  ds007                      158 docs — bold(118), T1w(20), inplaneT2(20)
  ds113b                     156 docs — bold(156)
  ds107                      147 docs — bold(98), T1w(49)


In [38]:
# Stats on RepetitionTime grouped by MagneticFieldStrength
response = client.search(
    index=INDEX_NAME,
    size=0,
    query={"exists": {"field": "RepetitionTime"}},
    aggs={
        "by_field_strength": {
            "terms": {"field": "MagneticFieldStrength"},
            "aggs": {
                "tr_stats": {"stats": {"field": "RepetitionTime"}}
            }
        }
    }
)

print("RepetitionTime stats by field strength:")
for bucket in response["aggregations"]["by_field_strength"]["buckets"]:
    stats = bucket["tr_stats"]
    print(f"\n  {bucket['key']}T ({bucket['doc_count']} scans):")
    for k, v in stats.items():
        if v is not None:
            print(f"    {k}: {v}")

RepetitionTime stats by field strength:

  3.0T (408 scans):
    count: 408
    min: 0.006727200001478195
    max: 8.0
    avg: 2.306276872555506
    sum: 940.9609640026465

  1.5T (15 scans):
    count: 15
    min: 0.010999999940395355
    max: 8.300000190734863
    avg: 3.756400093436241
    sum: 56.34600140154362


In [39]:
# Nested aggregation: Manufacturer → scanner model → count
response = client.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_manufacturer": {
            "terms": {"field": "Manufacturer", "missing": "(unknown)"},
            "aggs": {
                "by_model": {
                    "terms": {"field": "ManufacturersModelName", "missing": "(unknown)"}
                },
                "unique_subjects": {
                    "cardinality": {"field": "subject"}
                }
            }
        }
    }
)

print("Scanner distribution:\n")
print(f"  {'Manufacturer':20s} {'Docs':>6s} {'Subjects':>10s}  Models")
print(f"  {'-'*70}")
for bucket in response["aggregations"]["by_manufacturer"]["buckets"]:
    models = ", ".join(m["key"] for m in bucket["by_model"]["buckets"])
    print(
        f"  {bucket['key']:20s} {bucket['doc_count']:6d} {bucket['unique_subjects']['value']:10.0f}  {models}")

Scanner distribution:

  Manufacturer           Docs   Subjects  Models
  ----------------------------------------------------------------------
  (unknown)              3889         66  (unknown)
  Siemens                 364         24  TrioTim, (unknown), Prisma, Skyra, Skyra Singo, Avanto , Prisma_fit, High-Resolution Research Tomograph (HRRT, CTI/Siemens), Biograph_mMR, Investigational_Device_7T
  Philips                 162         16  Achieva
  GE                        7          4  Discovery MR750, DISCOVERY MR750, DISCOVERY_MR750
  GE MEDICAL SYSTEMS        1          1  GE Advance


In [40]:
# Histogram: distribution of MagneticFieldStrength across all scans
response = client.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "field_strength_hist": {
            "histogram": {
                "field": "MagneticFieldStrength",
                "interval": 0.5
            }
        }
    }
)

print("MagneticFieldStrength distribution (0.5T bins):")
for bucket in response["aggregations"]["field_strength_hist"]["buckets"]:
    if bucket["doc_count"] > 0:
        bar = "█" * bucket["doc_count"]
        print(f"  {bucket['key']:5.1f}T: {bar} ({bucket['doc_count']})")

MagneticFieldStrength distribution (0.5T bins):
    1.5T: ███████████████ (15)
    3.0T: █████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████ (505)
    6.5T: ██ (2)


---
## Exercise

Try writing your own queries:

1. Find all scans from dataset `ds000117` that have a `Manufacturer` field
2. Find BOLD scans at 3T with TR < 2.5s — which datasets do they come from?
3. Compute the average FlipAngle per dataset (using a nested aggregation)

*(Solutions hidden below — scroll down after trying!)*

In [ ]:
# YOUR CODE HERE — Exercise 1

In [ ]:
# YOUR CODE HERE — Exercise 2

In [ ]:
# YOUR CODE HERE — Exercise 3

<details>
<summary><b>Click for solutions</b></summary>

```python
# Exercise 1
client.search(index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"term": {"dataset": "ds000117"}},
                {"exists": {"field": "Manufacturer"}}
            ]
        }
    },
    size=10
)

# Exercise 2
client.search(index=INDEX_NAME,
    query={
        "bool": {
            "must": [
                {"term": {"suffix": "bold"}},
                {"term": {"MagneticFieldStrength": 3.0}}
            ],
            "filter": [{"range": {"RepetitionTime": {"lt": 2.5}}}]
        }
    },
    size=10
)

# Exercise 3
client.search(index=INDEX_NAME,
    size=0,
    query={"exists": {"field": "FlipAngle"}},
    aggs={
        "by_dataset": {
            "terms": {"field": "dataset"},
            "aggs": {
                "avg_flip_angle": {"avg": {"field": "FlipAngle"}}
            }
        }
    }
)
```
</details>

---
## Summary

You've learned:
- ✅ `match` and `multi_match` for full-text BM25 search
- ✅ `term` and `terms` for exact keyword matching
- ✅ `range` for numeric filtering
- ✅ `bool` queries to combine clauses with must/should/filter/must_not
- ✅ Aggregations: `terms`, `stats`, `cardinality`, `histogram`

**Next:** Open **Notebook 3** to unlock the power of **vector (semantic) search**.